# 你的第一个实验（社区版：Playwright 抓取 JS 站点）
### 请阅读本节。即便篇幅较长，这些内容也很重要，能帮你做好准备。

### 另外，请务必阅读 [README.md](../README.md)！更多资源见 [紫色课程资源页](https://edwarddonner.com/2024/11/13/llm-engineering-resources/)

## 练习目标（理念）

本笔记本在官方 Day 1「网站摘要器」基础上，额外演示：**用 Playwright 抓取依赖 JavaScript 渲染的现代网站**（例如 openai.com）。

到本课程结束时，你将构建由多个智能体协作的 Agentic AI 方案。别急——先从更小的事情开始。

目标：给定一个 URL，返回网站摘要——互联网版的《读者文摘》！！

开始之前，你应已完成 README 中链接的环境配置。

### 如果你刚接触 Notebook（Labs / Jupyter）

点击下方包含代码的「单元格」，按 **Shift+Enter** 执行。请从顶部开始，按顺序运行每一个单元格。

指南见 [Guides 文件夹](../guides/01_intro.ipynb)。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `openai.chat.completions.create(...)` |
| system / user 提示 | `system_prompt` + `user_prompt_prefix` |
| 网页抓取 | `fetch_website_contents`；进阶用 Playwright |
| Markdown 展示 | `display(Markdown(...))` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. `.env` 中准备好 `OPENAI_API_KEY`
3. Playwright 相关单元格需已安装 `playwright` 并执行过 `playwright install`

## 我随时可以帮忙

有问题可通过课程平台、邮件 ed@edwarddonner.com 或 LinkedIn 联系。也在尝试 X：[@edwarddonner](https://x.com/edwarddonner)。

## 更多故障排查

见 setup 文件夹中的 [troubleshooting](../setup/troubleshooting.ipynb)。

## 如果这些对你来说已经很熟悉

前几个实验可以快速过；后面会越来越深入。最终还会微调自己的 LLM。

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">请阅读——重要说明</h2>
            <span style="color:#900;">建议在观看讲座<strong>之后</strong>，自己仔细执行一遍。加 print 理解过程，再做自己的变体。有 Github 账号就用它展示变体——既是练习，也能向他人展示技能。</span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">这些代码是动态更新的资源</h2>
            <span style="color:#f71;">课程代码会定期更新；笔记本与视频可能不完全一致。请留意 Udemy「Announcements」中的邮件更新。</span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">这些练习的商业价值</h2>
            <span style="color:#181;">摘要是经典 GenAI 用例：新闻、财报、简历……学技术时想想如何在业务中落地。</span>
        </td>
    </tr>
</table>


### 如有需要，安装 Cursor 扩展

1. 从 **View** 菜单选择 **Extensions**
2. 搜索 Python
3. 点击由 `ms-python` 提供的 **Python**，若尚未安装则选 Install
4. 搜索 Jupyter
5. 点击由 `ms-toolsai` 提供的 **Jupyter**，若尚未安装则选 Install

### 接下来选择内核（Kernel）

点击右上角 **Select Kernel** → **Python Environments...** → 选择带推荐星标的 `.venv (Python 3.12.x)`。

有问题？前往 troubleshooting 笔记本。

### 注意：每个笔记本都需要单独设置一次内核


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从本课 scraper 模块导入抓取函数：给定 URL，返回网站正文文本
from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具：在笔记本里漂亮地显示 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI

# 若本格报错（ModuleNotFoundError 等），请先去 troubleshooting 笔记本排查环境


# 连接到 OpenAI（或 Ollama）

下一格会加载 `.env` 中的环境变量，并检查 `OPENAI_API_KEY` 是否可用。

若想用免费的本地 **Ollama**，请参阅 README「付费 API 的免费替代方案」；完整示例见 solutions 文件夹的 `day1_with_ollama.ipynb`。

## 遇到问题？排查清单

- 出现 `NameError`：是否从上到下跑过所有单元格？见 Python 基础指南。
- 仍不行：打开 [troubleshooting](../setup/troubleshooting.ipynb) 做分步诊断。
- 或联系 ed@edwarddonner.com。

对 API 费用担心？README 有说明——本练习成本通常很低；也可用 Ollama（Day 2 会讲）。


In [ ]:
# ========== 环境变量：加载并校验 OpenAI API Key ==========

# override=True：用 .env 中的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境变量读取密钥；不要把真实 key 硬编码进笔记本
api_key = os.getenv('OPENAI_API_KEY')

# 做几项「形状检查」：缺 key / 前缀不对 / 首尾空白——都是常见踩坑

if not api_key:
    # 完全没读到：多半是没建 .env 或变量名写错
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    # OpenAI 项目密钥通常以 sk-proj- 开头；不对则可能拷错了别的 key
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    # 复制粘贴时容易带上空格/Tab
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    # 形状看起来正常（真正能否调用还要看网络与额度）
    print("API key found and looks good so far!")




# 先快速调用一次前沿模型，当作预热预览！


In [ ]:
# ========== 预览：构造发给模型的 messages 列表 ==========

# 用户可见的第一句话（发给模型的内容保持英文；这是可运行、影响回答的字符串）
message = "Hello, GPT! This is my first ever message to you! Hi!"

# Chat Completions 约定：每条消息是 {"role": ..., "content": ...}
# 这里只有 user，没有 system——最简单的一次对话
messages = [{"role": "user", "content": message}]

# 在笔记本里直接写变量名：会显示该对象，方便确认结构
messages




In [ ]:
# ========== 第一次真正调用 OpenAI Chat Completions ==========

# 创建客户端：默认从环境变量 OPENAI_API_KEY 取密钥
openai = OpenAI()

# create：发起一次聊天补全；model id 保持原样（可运行标识，不翻译）
response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
# choices[0].message.content：取第一条候选回复的文本正文
response.choices[0].message.content


## 开始我们的第一个项目：网站摘要器


In [ ]:
# ========== 试一下网页抓取工具 ==========

# fetch_website_contents：HTTP 拉取页面并抽出可读文本（对纯静态页友好）
ed = fetch_website_contents("https://edwarddonner.com")
# 打印抓到的正文，确认抓取链路通了
print(ed)


## 提示类型（Prompt Types）

你可能已知道——若还不知道，很快会非常熟悉！

像 GPT 这类模型，训练时就约定以特定方式接收指令。它们期望收到：

- **系统提示（system prompt）**：告诉模型「你在执行什么任务、用什么语气」
- **用户提示（user prompt）**：对话的起点——模型要回复的具体内容


In [ ]:
# ========== 定义 system prompt：定人设与输出格式 ==========

# 可稍后实验：把最后一句改成「用西班牙语以 markdown 回复」等
# 注意：发给模型的指令字符串保持英文，翻译会改变模型行为

system_prompt = """
You are a snarkyassistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""


In [ ]:
# ========== 定义 user prompt 前缀：后面会拼接网站正文 ==========

# 前缀说明「下面是网页内容，请摘要」；真正的正文在调用时追加在后面
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


## 消息（Messages）结构

OpenAI API（以及许多兼容 API）期望消息是如下结构的列表：

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```

下面两格先做一个很简单的调用预览——还不会动用最强模型。


In [ ]:
# ========== 最小可运行示例：system + user → 一次补全 ==========

# 两条消息：system 定助手风格，user 放具体问题
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

# 用较小/较快的模型做算术预热（model id 保持原样）
response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
# 取出模型回复文本
response.choices[0].message.content


## 用函数为 GPT-4.1-mini 组装「有用的 messages」


In [ ]:
# ========== messages_for：把「网站正文」打包成 API 需要的 messages ==========

# 返回值结构与上一格示例相同：一条 system + 一条 user
def messages_for(website):
    return [
        # system：人设与格式（前面定义的 system_prompt）
        {"role": "system", "content": system_prompt},
        # user：前缀说明 + 抓取到的网站正文
        {"role": "user", "content": user_prompt_prefix + website}
    ]


In [ ]:
# ========== 试调用：看组装出的 messages 长什么样 ==========

# 传入前面抓到的 edwarddonner.com 正文；可再换别的网站试
messages_for(ed)


## 拼起来：OpenAI API 其实非常简单！


In [ ]:
# ========== summarize：抓取 URL → 组 messages → 调 API → 返回摘要文本 ==========

def summarize(url):
    # 1) 抓取网页正文
    website = fetch_website_contents(url)
    # 2) 调用 Chat Completions；model 用 gpt-4.1-mini（标识符保持原样）
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    # 3) 返回助手回复正文
    return response.choices[0].message.content


In [ ]:
# 对课程站点试一次 summarize（会真实调用 API）
summarize("https://edwarddonner.com")


In [ ]:
# ========== display_summary：摘要后再用 Markdown 漂亮展示 ==========

def display_summary(url):
    # 先拿到纯文本摘要
    summary = summarize(url)
    # 在 Jupyter 里渲染为 Markdown（标题、列表会排版）
    display(Markdown(summary))


In [ ]:
# 端到端演示：抓取 → 摘要 → Markdown 展示
display_summary("https://edwarddonner.com")


# 再多试几个网站

注意：这种简单抓取**只适合**「服务器直接返回可读 HTML」的站点。

用 JavaScript 渲染的站点（例如许多 React 应用）可能几乎抓不到正文。社区贡献里有 Selenium / Playwright 方案；本笔记本后半部分就是 Playwright 进阶。

受 CloudFront 等防护的站点可能返回 403——感谢 Andy J 指出。

但很多网站仍然能正常工作！


In [ ]:
# 试 CNN（新闻站；能否成功取决于对方反爬与页面结构）
display_summary("https://cnn.com")


In [ ]:
# 试 Anthropic 官网
display_summary("https://anthropic.com")


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">你第一次体验了调用前沿模型的 Cloud API。摘要是经典 GenAI 用例：新闻、财报、求职信……想想如何在自己的业务里做原型。</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">继续之前——现在就动手</h2>
            <span style="color:#900;">用下面单元格做你自己的简单商业示例。仍可用摘要：例如给一封邮件正文建议简短主题行——这正是商务邮件工具常见能力。</span>
        </td>
    </tr>
</table>


In [ ]:
# ========== 练习：邮件主题行建议（商业小用例）==========

# 第 1 步：创建提示——发给模型的指令/正文保持英文（影响行为，不翻译）

system_prompt = "You are a helpful assistant that can summarize an email and suggest a subject line."
email  = """Testing applications is essential to the development lifecycle, but LLM systems are non-deterministic – you can’t always predict how they will behave.

Add multi-turn interactions and tool-calling agents, and testing agents becomes even more complex than traditional software testing.

To address this challenge, LangSmith provides a comprehensive platform for agent engineering that helps teams use live production data for continuous testing and improvement.

In our new quickstart course, LangSmith Essentials, you’ll learn how to:

Trace your agent step by step
Evaluate it using production data
Improve prompts with real metrics (not just “vibes”)
Deploy with a single click using LangSmith Deployment
Here’s a preview:
"""
# user_prompt：约束主题行要短、专业、单行；再拼上邮件正文
user_prompt = """
    Keep the subject line short and concise and professional. Do not provide summary. The subject line should be single line and must have the core-purpose
    Keep any relevant nouns that are present in the email.
    """ + email

# 第 2 步：创建 messages 列表（system + user）

messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}] # fill this in

# 第 3 步：调用 OpenAI（model id 保持原样）
response = openai.chat.completions.create(model="gpt-5", messages=messages)   

# 第 4 步：打印模型给出的主题行
print(response.choices[0].message.content)


## 额外练习：喜欢网页抓取的同学看这里

你会发现：对 `display_summary("https://openai.com")` 往往**不行**——因为 OpenAI 官网大量依赖 JavaScript 渲染。

常见解法：Selenium、Playwright 等——在真实浏览器里渲染页面再抽取文本。若你有相关经验，可以改进抓取逻辑。社区贡献文件夹里已有同学分享的方案。

本笔记本接下来演示 **Playwright** 路径。


In [ ]:
# 对照组：简单 requests 抓取对 openai.com 通常失败或内容残缺
display_summary("https://openai.com")


**为什么在 JavaScript 网站上用 Playwright？**

像 openai.com 这类现代站点，用 React / Next.js 等在浏览器里**动态渲染**内容。`requests` + `BeautifulSoup` 往往只能拿到初始 HTML，缺少 JS 渲染后的正文。

**Playwright** 会启动真实浏览器（Chromium / Firefox / WebKit），执行 JavaScript，等页面呈现后再抓取——从而拿到完整内容。它比 Selenium 更现代，异步支持更好，也便于跨浏览器。

当你看到「请启用 JavaScript」，或用普通 HTTP 抓取结果为空时，就该考虑 Playwright。


In [ ]:
# ========== Playwright 异步抓取：在真实浏览器里渲染页面 ==========

# 目标 URL（依赖 JS 渲染的典型例子）
url = "https://openai.com"
# asyncio：Python 异步运行时（本格在 Jupyter 里用 async with / await）
import asyncio
# async_playwright：异步 Playwright API
from playwright.async_api import Playwright, async_playwright
# 同步 API 也导入了（本格主要用异步路径；逻辑保持原样不删）
from playwright.sync_api import Playwright, sync_playwright

async def run(playwright: Playwright, url: str):
    # headless=False：有界面模式，方便观察浏览器行为（无头可改为 True）
    browser = await playwright.chromium.launch(headless=False)
    # 新建页面并设视口大小
    page = await browser.new_page(viewport={"width": 1600, "height": 900})
    
    # 导航到目标 URL（会执行页面上的 JS）
    await page.goto(url)
    
    # 读取浏览器里的 document.title
    title = await page.title()
    
    # 抽取 body 内可见文本（渲染后的）
    body_text = await page.inner_text('body')
    
    # 收集 h1/h2/h3 标题文本
    headings = await page.locator('h1, h2, h3').all_text_contents()
    
    # 用 evaluate_all 在页面上下文中映射出链接 text/href
    links = await page.locator('a').evaluate_all(
        "elements => elements.map(e => ({text: e.textContent, href: e.href}))"
    )
    
    # 完整 HTML 快照（当前未放进返回值，需要时可取消下面注释）
    html = await page.content()

     # 再在页面里跑一段 JS，结构化抽取 headings / paragraphs / links
    data = await page.evaluate('''() => {
        return {
            headings: Array.from(document.querySelectorAll('h1, h2, h3')).map(h => h.textContent),
            paragraphs: Array.from(document.querySelectorAll('p')).map(p => p.textContent),
            links: Array.from(document.querySelectorAll('a')).map(a => ({
                text: a.textContent,
                href: a.href
            }))
        }
    }''')
    
    # 用完关闭浏览器，释放资源
    await browser.close()
    
    # 打包返回；html 默认注释掉以免输出过大
    return {
        "url": url,
        "title": title,
        "body_text": body_text,
        "headings": headings,
        "links": links,
        "structured_data": data,
        # "html": html # 如果需要原始 HTML，请取消注释
    }

# 启动 Playwright，抓取并打印 body 文本
async with async_playwright() as playwright:
    result = await run(playwright, url=url)
    print(result['body_text'])






In [ ]:
# ========== 智能抓取：先普通 HTTP，遇到「需 JS」再走 Playwright ==========

# BeautifulSoup：解析 HTML
from bs4 import BeautifulSoup
# requests：同步 HTTP 客户端
import requests

# 伪装成常见浏览器的 User-Agent，降低被直接拒绝的概率
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

async def get_js_web_content(url):
    """异步：用上一格的 run() 经 Playwright 取 body 文本。"""
    async with async_playwright() as playwright:
        result = await run(playwright, url=url)
        return result['body_text']


async def fetch_website_contents_new(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    # 先走轻量路径：普通 GET + BeautifulSoup
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    
    # 若页面提示需要启用 JS，则升级到 Playwright
    # 打印(soup.body.get_text().lower())
    if "enable javascript" in soup.body.get_text().lower():
        response = await get_js_web_content(url)
        return response
    
    # 否则清理 script/style 等无关标签，抽出正文
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    # 截断到 2000 字符，控制送给模型的上下文长度与费用
    return (title + "\n\n" + text)[:2_000]

async def summarize_new(url):
    """异步摘要：新抓取函数 + 原 messages_for + Chat Completions。"""
    website = await fetch_website_contents_new(url)
    print(website)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

# 使用 Markdown 漂亮展示摘要

async def display_summary_new(url):
    """端到端：异步抓取摘要并 display。"""
    summary = await summarize_new(url)
    print('Summary is', summary)
    display(Markdown(summary))


In [ ]:
# 对 CNN 试新版异步摘要（可能走普通 HTTP，不一定需要 Playwright）
await display_summary_new("https://cnn.com")


In [ ]:
# 对 openai.com 试新版：通常会检测到需 JS，从而走 Playwright
await display_summary_new("https://openai.com")


# 分享你的代码

欢迎把改进分享出来！社区贡献文件夹里已有同学的 Selenium 等实现。若要加入该文件夹，请提交包含新版本的 Pull Request。

若不熟 git，可让 GPT 指导如何开 PR。专业提示：分享前可用 Edit → Clear All Outputs 清输出，得到更干净的笔记本（**注意：本教学注释任务禁止清 outputs**；这是给「对外分享」的建议）。

参考说明：  
https://chatgpt.com/share/677a9cb5-c64c-8012-99e0-e06e88afd293
